# Explorar las curvas guardadas / Inspect saved curves

Este notebook **solo lee el CSV de salida**. No consulta inputs propios ni archivos EEX, no rellena precios y no modifica los datos. Las vistas separan **una fecha**, **medias por rango** y **evolución de una entrega fija**.

Guías: [Español](../NOTEBOOK.es.md) · [English](../NOTEBOOK.md). Select a full product/region/unit identity; paired averages use equal daily weights and keep zero/negative prices.

Ejecuta las celdas en orden. Si copias el proyecto a otro ordenador, configura abajo la ruta del CSV o la del TOML. No hacen falta los inputs originales.

La celda siguiente está preparada para el **CSV sintético de prueba de este ordenador**. Ese archivo no está incluido en GitHub. En otro ordenador, cambia `OUTPUT_PATH` por tu `filled_history.csv`; tienes ejemplos comentados. / The next cell uses this computer's synthetic validation CSV, which is not included in GitHub. Change `OUTPUT_PATH` on another computer.


In [ ]:
# This computer: open the repository folder in VS Code and use its .venv kernel.
PROJECT_ROOT = None  # Discover the repository from the root or notebooks/ folder.
CONFIG_PATH = None  # Use the root config.toml if OUTPUT_PATH is not specified.
OUTPUT_PATH = "output/notebook_validation/filled_history.csv"
# This CSV contains synthetic validation data; it is local and not shipped in GitHub.
# Relative OUTPUT_PATH values are resolved from PROJECT_ROOT, not notebooks/.

# Other computer: replace the assignments above with your actual locations.
# PROJECT_ROOT = r"D:/Projects/VWAPS"  # Folder containing pyproject.toml and src/.
# CONFIG_PATH = None
# OUTPUT_PATH = r"D:/Results/VWAPS/filled_history.csv"

# Alternative: use output/ inside the copied repository.
# OUTPUT_PATH = "output/filled_history.csv"
# Or let the TOML select paths.output_dir: CONFIG_PATH = None; OUTPUT_PATH = None.


In [ ]:
from pathlib import Path
import sys

search_from = Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else Path.cwd().resolve()
ROOT = next(
    (path for path in (search_from, *search_from.parents)
     if (path / "pyproject.toml").is_file() and (path / "src" / "vwaps").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Indica PROJECT_ROOT: la carpeta que contiene pyproject.toml y src/vwaps.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from vwaps.visualization import (
    available_curves, contract_evolution, coverage_metrics,
    load_output, paired_means, resolve_output_path, select_curve,
)

CSV_PATH = resolve_output_path(ROOT, config_path=CONFIG_PATH, output_path=OUTPUT_PATH)
try:
    DATA = load_output(CSV_PATH)
except FileNotFoundError:
    DATA = pd.DataFrame()
    print(f"No se encuentra el CSV: {CSV_PATH}")
    print("Indica OUTPUT_PATH en la celda anterior y vuelve a ejecutar desde esta celda.")
else:
    print(f"CSV: {CSV_PATH}")
    if DATA.empty:
        print("El archivo no contiene filas. Selecciona otro output o genera el relleno fuera de este notebook.")
    else:
        print(f"{len(DATA):,} observaciones únicas · {len(available_curves(DATA))} identidades")
        print(f"Etiquetas equivalentes deduplicadas: {DATA.attrs.get('alias_rows_removed', 0)}")
        print("EEX se muestra tal como figura en eex_settle; su antigüedad se consulta en eex_asof.")


In [ ]:
# Optional interactive dependencies are imported only by this notebook.
try:
    import ipywidgets as widgets
    import plotly.graph_objects as go
    from IPython.display import clear_output, display
except ImportError as exc:
    raise ImportError(
        "Faltan dependencias del notebook. Desde la raíz: uv sync --group notebook; "
        "después: uv run --group notebook jupyter lab notebooks/inspect_curves.ipynb"
    ) from exc

METRIC_LABELS = {
    "n_observations": "Filas únicas", "n_dates": "Fechas", "n_price": "Con precio",
    "n_eex": "Con EEX", "n_paired": "Pares precio/EEX", "n_missing_price": "Sin precio",
    "n_original": "Originales finales", "n_estimated": "Estimados finales",
    "n_adjusted_original": "Originales ajustados (subgrupo)", "price_coverage": "Cobertura de filas",
}

def show_metrics(frame):
    metrics = coverage_metrics(frame)
    metrics["price_coverage"] = (
        f"{100 * metrics['price_coverage']:.1f}%" if len(frame) else "Sin datos"
    )
    display(pd.DataFrame([metrics]).rename(columns=METRIC_LABELS))

def show_details(frame):
    fields = [
        "reference_date", "tenor", "tenor_aliases", "kind", "delivery_start", "delivery_end",
        "price", "eex_settle", "eex_asof", "eex_age_days", "source", "data_origin",
        "own_vwap", "price_before_shape", "shape_mode", "shape_status", "shape_adjustment", "flag",
    ]
    display(frame[[name for name in fields if name in frame]].reset_index(drop=True))

def add_price_traces(figure, frame, x_column, label="", show_before=False, show_own=False, connect=True):
    mode = "lines+markers" if connect else "markers"
    specifications = [
        ("price", "Precio final", "#1261a0", "solid", mode),
        ("eex_settle", "Referencia EEX", "#777777", "dash", mode),
    ]
    if show_before and "price_before_shape" in frame:
        specifications.append(("price_before_shape", "Antes de shape", "#d68100", "dot", mode))
    if show_own and "own_vwap" in frame:
        specifications.append(("own_vwap", "Propio observado", "#258a45", "solid", "markers"))
    hover = [
        f"{row.tenor} · {row.kind} · {row.delivery_start:%Y-%m-%d} / {row.delivery_end:%Y-%m-%d}"
        for row in frame.itertuples()
    ]
    for column, name, color, dash, trace_mode in specifications:
        if not frame[column].notna().any():
            continue
        figure.add_trace(go.Scatter(
            x=frame[x_column], y=frame[column], name=f"{label}{name}",
            mode=trace_mode, connectgaps=False,
            line={"color": color, "dash": dash},
            marker={"symbol": "x" if column == "own_vwap" else "circle", "size": 8},
            text=hover, hovertemplate="%{text}<br>%{x}<br>%{y:.6f}<extra>%{fullData.name}</extra>",
        ))

def finish_figure(figure, title, unit, x_title):
    figure.update_layout(
        title=title, template="plotly_white", height=500,
        xaxis_title=x_title, yaxis_title=unit or "Unidad no informada",
        hovermode="closest", legend={"orientation": "h", "y": -0.2},
        margin={"b": 130},
    )
    display(figure)


In [ ]:
def build_browser(data):
    identities = available_curves(data)
    identity = widgets.Dropdown(
        options=[(f"{p} | región={r!r} | unidad={u!r}", (p, r, u)) for p, r, u in identities],
        description="Curva:", layout=widgets.Layout(width="95%"),
    )
    kind = widgets.Dropdown(description="Tipo:")
    reference = widgets.Dropdown(description="Fecha:")
    show_before = widgets.Checkbox(value=False, description="Antes de shape", disabled="price_before_shape" not in data)
    show_own = widgets.Checkbox(value=True, description="Propios observados", disabled="own_vwap" not in data)
    start = widgets.DatePicker(description="Desde:")
    end = widgets.DatePicker(description="Hasta:")
    alignment = widgets.Dropdown(
        options=[("Entrega absoluta (recomendado)", "delivery"), ("Tenor relativo: mezcla entregas", "tenor")],
        value="delivery", description="Alinear:", layout=widgets.Layout(width="480px"),
    )
    compare = widgets.Button(description="Comparar rango", button_style="primary")
    contract = widgets.Dropdown(description="Entrega:", layout=widgets.Layout(width="95%"))
    daily_output, range_output, evolution_output = widgets.Output(), widgets.Output(), widgets.Output()
    state = {"updating": False}

    def selected_scope():
        return select_curve(data, identity.value, kind=kind.value)

    def daily_view(change=None):
        if state["updating"]:
            return
        with daily_output:
            clear_output(wait=True)
            if reference.value is None:
                print("No hay fechas disponibles para esta selección.")
                return
            selected = select_curve(data, identity.value, kind=kind.value, reference_date=reference.value)
            if selected.empty:
                print("No hay filas para esa fecha.")
                return
            print("Entrega: [inicio, fin). La cobertura cuenta filas guardadas, no contratos ausentes del CSV.")
            if kind.value is None:
                print("Tipos mezclados: solo marcadores; meses y trimestres no se unen en una línea.")
            if "eex_asof" in selected:
                stale = selected["eex_age_days"].gt(0).sum()
                unknown = selected["eex_asof"].isna().sum()
                print(f"EEX anterior a la fecha: {stale} filas · fecha EEX no informada: {unknown}")
            show_metrics(selected)
            figure = go.Figure()
            for name, group in selected.groupby("kind", sort=True):
                group = group.sort_values("delivery_start")
                prefix = f"{name} · " if kind.value is None else ""
                add_price_traces(
                    figure, group, "delivery_start", prefix, show_before.value, show_own.value,
                    connect=kind.value is not None,
                )
            finish_figure(figure, f"{identity.value[0]} · {reference.value}", identity.value[2], "Inicio de entrega")
            show_details(selected)

    def range_view(change=None):
        if state["updating"]:
            return
        with range_output:
            clear_output(wait=True)
            if start.value is None or end.value is None:
                print("Selecciona ambas fechas del rango.")
                return
            if start.value > end.value:
                print("La fecha inicial debe ser anterior o igual a la final.")
                return
            selected = select_curve(data, identity.value, kind=kind.value, start=start.value, end=end.value)
            if selected.empty:
                print("No hay observaciones en este rango.")
                return
            if alignment.value == "tenor":
                print("AVISO: M+1, Q+1, etc. cambian de entrega al hacer roll. Esta vista mezcla contratos.")
            print("Medias con peso diario uniforme. En cada punto se usan solo las mismas fechas con precio Y EEX finitos.")
            print("n_paired muestra las fechas usadas; n_price/n_eex incluyen también observaciones sin pareja.")
            print("Distintos puntos pueden tener distintas fechas y tamaños de muestra.")
            show_metrics(selected)
            means = paired_means(
                data, identity.value, kind=kind.value, start=start.value, end=end.value, alignment=alignment.value,
            )
            if means.empty:
                print("Sin puntos comparables.")
                return
            x_column = "delivery_start" if alignment.value == "delivery" else "tenor"
            figure = go.Figure()
            mode = "lines+markers" if kind.value is not None else "markers"
            for name, group in means.groupby("kind", sort=False):
                prefix = f"{name} · " if kind.value is None else ""
                for column, title, color, dash in (
                    ("price_mean", "Media precio", "#1261a0", "solid"),
                    ("eex_mean", "Media EEX emparejada", "#777777", "dash"),
                ):
                    figure.add_trace(go.Scatter(
                        x=group[x_column], y=group[column], name=prefix + title, mode=mode,
                        connectgaps=False, line={"color": color, "dash": dash},
                        customdata=group[["n_paired", "n_observations", "n_delivery_periods"]].to_numpy(),
                        hovertemplate=(
                            "%{x}<br>%{y:.6f}<br>Fechas emparejadas: %{customdata[0]}"
                            "<br>Observaciones: %{customdata[1]}<br>Entregas: %{customdata[2]}"
                            "<extra>%{fullData.name}</extra>"
                        ),
                    ))
            finish_figure(
                figure, f"Medias · {start.value} a {end.value}", identity.value[2],
                "Entrega absoluta" if alignment.value == "delivery" else "Tenor relativo",
            )
            display(means.reset_index(drop=True))

    def evolution_view(change=None):
        if state["updating"]:
            return
        with evolution_output:
            clear_output(wait=True)
            if contract.value is None:
                print("No hay entregas disponibles.")
                return
            contract_kind, delivery_start, delivery_end = contract.value
            selected = contract_evolution(data, identity.value, contract_kind, delivery_start, delivery_end)
            print("Misma entrega absoluta en todas las fechas; la etiqueta relativa puede cambiar.")
            print("Los valores EEX pueden proceder de publicaciones anteriores: consulta eex_asof y eex_age_days.")
            show_metrics(selected)
            figure = go.Figure()
            add_price_traces(figure, selected, "reference_date", show_before=show_before.value, show_own=show_own.value)
            finish_figure(
                figure, f"{contract_kind} · {delivery_start:%Y-%m-%d} / {delivery_end:%Y-%m-%d}",
                identity.value[2], "Fecha de referencia",
            )
            show_details(selected)

    def refresh_controls(change=None, refresh_kinds=False):
        state["updating"] = True
        if refresh_kinds:
            kinds = sorted(select_curve(data, identity.value)["kind"].unique())
            kind.options = [(name, name) for name in kinds] + [("Todos (solo marcadores)", None)]
            kind.value = "Month" if "Month" in kinds else kinds[0]
        selected = selected_scope()
        dates = sorted(selected["reference_date"].dt.date.unique())
        reference.options = [(str(day), day) for day in dates]
        reference.value = dates[-1] if dates else None
        start.value = dates[0] if dates else None
        end.value = dates[-1] if dates else None
        periods = selected[["kind", "delivery_start", "delivery_end"]].drop_duplicates().sort_values(
            ["kind", "delivery_start", "delivery_end"]
        )
        contract.options = [
            (f"{name} · {first:%Y-%m-%d} / {last:%Y-%m-%d}", (name, first, last))
            for name, first, last in periods.itertuples(index=False, name=None)
        ]
        state["updating"] = False
        daily_view()
        range_view()
        evolution_view()

    identity.observe(lambda change: refresh_controls(refresh_kinds=True), names="value")
    kind.observe(lambda change: None if state["updating"] else refresh_controls(), names="value")
    reference.observe(daily_view, names="value")
    show_before.observe(daily_view, names="value")
    show_own.observe(daily_view, names="value")
    show_before.observe(evolution_view, names="value")
    show_own.observe(evolution_view, names="value")
    contract.observe(evolution_view, names="value")
    compare.on_click(range_view)
    alignment.observe(range_view, names="value")
    tabs = widgets.Tab(children=[
        widgets.VBox([reference, daily_output]),
        widgets.VBox([widgets.HBox([start, end]), alignment, compare, range_output]),
        widgets.VBox([contract, evolution_output]),
    ])
    for i, title in enumerate(("Una fecha", "Medias por rango", "Evolución de una entrega")):
        tabs.set_title(i, title)
    controls = widgets.VBox([identity, kind, widgets.HBox([show_before, show_own]), tabs])
    refresh_controls(refresh_kinds=True)
    return controls

if DATA.empty:
    print("No se crean selectores hasta cargar un CSV con datos. Ajusta OUTPUT_PATH y vuelve a ejecutar.")
else:
    BROWSER = build_browser(DATA)
    display(BROWSER)


### Cómo interpretar las vistas

- **Una fecha:** el precio final y su referencia `eex_settle`; no es una nueva consulta a EEX. `own_vwap` es el propio del periodo que guardó el motor, que puede agregar filas originales equivalentes.
- **Medias por rango:** cada fecha emparejada pesa lo mismo, independientemente de volumen u horas. Las horas corresponden a otra operación: agregar entregas dentro de un trimestre/año.
- **Evolución:** sigue un contrato absoluto; un salto puede deberse a nueva información, cambio de ruta, shape o antigüedad EEX. El gráfico no identifica por sí solo la causa.
- Shape es opcional. En `audit`, `price` sigue siendo el precio publicado sin aplicar la propuesta; consulta `shape_status`. La curva naranja opcional muestra `price_before_shape`, cuando exista.
- Las filas de origen `missing` nunca cuentan como precio cero. La cobertura se refiere a filas existentes del CSV, no a todos los contratos que podrían faltar por completo.
- Contratos con etiquetas equivalentes cuentan una vez por fecha, identidad, tipo y entrega. Un conflicto de valores detiene la carga para que se revise; nunca se promedian aliases contradictorios.

**English:** this notebook is read-only. It distinguishes point-in-time curves, paired daily means over a date range, and the evolution of a fixed absolute delivery contract. See [NOTEBOOK.md](../NOTEBOOK.md) for setup, scope and limitations.
